In [ ]:
import io
import pandas as pd

In [5]:
!pip install duckdb
!pip install shap


In [6]:
import seaborn as sns
import duckdb

In [9]:
# Laadime kohaliku CSV faili DuckDB abil otse andmetabelisse
# (Veendu, et squirrel_data.csv fail on samas kaustas sinu Notebookiga)
squirrel = duckdb.read_csv("Squirrel_Data_Raw.csv").df()

# Kuvame puhastamiseks kättesaadavad andmed
squirrel

,X,Y,Unique Squirrel ID,Hectare,Shift,Date,Hectare Squirrel Number,Age,Primary Fur Color,Highlight Fur Color,...,Kuks,Quaas,Moans,Tail flags,Tail twitches,Approaches,Indifferent,Runs from,Other Interactions,Lat/Long
0,-73.956134,40.794082,37F-PM-1014-03,37F,PM,10142018,3,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,NaN,POINT (-73.9561344937861 40.7940823884086)
1,-73.968857,40.783783,21B-AM-1019-04,21B,AM,10192018,4,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,NaN,POINT (-73.9688574691102 40.7837825208444)
2,-73.974281,40.775534,11B-PM-1014-08,11B,PM,10142018,8,NaN,Gray,NaN,...,False,False,False,False,False,False,False,False,NaN,POINT (-73.97428114848522 40.775533619083)
3,-73.959641,40.790313,32E-PM-1017-14,32E,PM,10172018,14,Adult,Gray,NaN,...,False,False,False,False,False,False,False,True,NaN,POINT (-73.9596413903948 40.7903128889029)
4,-73.970268,40.776213,13E-AM-1017-05,13E,AM,10172018,5,Adult,Gray,Cinnamon,...,False,False,False,False,False,False,False,False,NaN,POINT (-73.9702676472613 40.7762126854894)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3018,-73.963943,40.790868,30B-AM-1007-04,30B,AM,10072018,4,Adult,Gray,NaN,...,False,False,False,False,False,False,False,True,NaN,POINT (-73.9639431360458 40.7908677445466)
3019,-73.970402,40.782560,19A-PM-1013-05,19A,PM,10132018,5,Adult,Gray,White,...,False,False,False,False,False,False,True,False,NaN,POINT (-73.9704015859639 40.7825600069973)
3020,-73.966587,40.783678,22D-PM-1012-07,22D,PM,10122018,7,Adult,Gray,"Black, Cinnamon, White",...,False,False,False,False,False,False,True,False,NaN,POINT (-73.9665871993517 40.7836775064883)
3021,-73.963994,40.789915,29B-PM-1010-02,29B,PM,10102018,2,NaN,Gray,"Cinnamon, White",...,False,False,False,False,False,False,True,False,NaN,POINT (-73.9639941227864 40.7899152327912)


In [10]:
# 1. LOOME DIMENSIOONID (DIM TABLES)

# Dim 1: Orava profiil (Vanus ja karva värvused)
dim_squirrel = duckdb.sql("""
    SELECT 
        ROW_NUMBER() OVER () AS Squirrel_Profile_ID,
        COALESCE(Age, 'Unknown') AS Age,
        COALESCE("Primary Fur Color", 'Unknown') AS Primary_Fur_Color,
        COALESCE("Highlight Fur Color", 'Unknown') AS Highlight_Fur_Color
    FROM (
        SELECT DISTINCT Age, "Primary Fur Color", "Highlight Fur Color" 
        FROM squirrel
    )
""").df()

# Dim 2: Asukoht (Hektar ja spetsiifilised asukohad pargis)
dim_location = duckdb.sql("""
    SELECT 
        ROW_NUMBER() OVER () AS Location_ID,
        Hectare,
        COALESCE(Location, 'Unknown') AS Location,
        COALESCE("Specific Location", 'Unknown') AS Specific_Location
    FROM (
        SELECT DISTINCT Hectare, Location, "Specific Location" 
        FROM squirrel
    )
""").df()

# Dim 3: Aeg (Kuupäevad, vahetused ja detailsed ajatükid filtrite jaoks)
dim_time = duckdb.sql("""
    SELECT 
        ROW_NUMBER() OVER () AS Time_ID,
        parsed_date AS Date,
        Shift,
        EXTRACT(year FROM parsed_date) AS Year,
        EXTRACT(month FROM parsed_date) AS Month,
        EXTRACT(day FROM parsed_date) AS Day
    FROM (
        SELECT DISTINCT 
            CAST(strptime(CAST(Date AS VARCHAR), '%m%d%Y') AS DATE) AS parsed_date, 
            Shift 
        FROM squirrel
    )
""").df()


# 2. LOOME FAKTITABELI (FACT TABLE)


# Seome dimensioonitabelite võtmed (ID-d) algsete vaatlusandmetega kokku
fact_observations = duckdb.sql("""
    SELECT 
        s."Unique Squirrel ID",
        ds.Squirrel_Profile_ID,
        dl.Location_ID,
        dt.Time_ID,
        s.X,
        s.Y,
        TRY_CAST(s."Hectare Squirrel Number" AS INTEGER) AS Hectare_Squirrel_Number,
        CASE WHEN LOWER(CAST(s.Running AS VARCHAR)) = 'true' THEN True ELSE False END AS Running,
        CASE WHEN LOWER(CAST(s.Chasing AS VARCHAR)) = 'true' THEN True ELSE False END AS Chasing,
        CASE WHEN LOWER(CAST(s.Climbing AS VARCHAR)) = 'true' THEN True ELSE False END AS Climbing,
        CASE WHEN LOWER(CAST(s.Eating AS VARCHAR)) = 'true' THEN True ELSE False END AS Eating,
        CASE WHEN LOWER(CAST(s.Foraging AS VARCHAR)) = 'true' THEN True ELSE False END AS Foraging,
        COALESCE(s."Other Activities", 'None') AS Other_Activities,
        COALESCE(s."Other Interactions", 'None') AS Other_Interactions
    FROM squirrel s
    LEFT JOIN dim_squirrel ds 
        ON COALESCE(s.Age, 'Unknown') = ds.Age 
       AND COALESCE(s."Primary Fur Color", 'Unknown') = ds.Primary_Fur_Color
       AND COALESCE(s."Highlight Fur Color", 'Unknown') = ds.Highlight_Fur_Color
    LEFT JOIN dim_location dl 
        ON s.Hectare = dl.Hectare 
       AND COALESCE(s.Location, 'Unknown') = dl.Location
       AND COALESCE(s."Specific Location", 'Unknown') = dl.Specific_Location
    LEFT JOIN dim_time dt 
        ON CAST(strptime(CAST(s.Date AS VARCHAR), '%m%d%Y') AS DATE) = dt.Date 
       AND s.Shift = dt.Shift
""").df()

# 3. KONTROLLIME TULEMUSI
print("=== Täht-skeem on edukalt loodud! ===")
print(f"Ridu faktitabelis: {len(fact_observations)}")
print(f"Ridu oravaprofiilide tabelis (dim_squirrel): {len(dim_squirrel)}")
print(f"Ridu asukohtade tabelis (dim_location): {len(dim_location)}")

=== Täht-skeem on edukalt loodud! ===
Ridu faktitabelis: 3023
Ridu oravaprofiilide tabelis (dim_squirrel): 49
Ridu asukohtade tabelis (dim_location): 992


In [11]:
# Salvestame puhastatud mudeli failidena Streamliti jaoks
fact_observations.to_csv("fact_observations.csv", index=False)
dim_squirrel.to_csv("dim_squirrel.csv", index=False)
dim_location.to_csv("dim_location.csv", index=False)
dim_time.to_csv("dim_time.csv", index=False)